qwen

In [ ]:
import json
import asyncio
from tqdm import tqdm
from ollama import AsyncClient
import os


import nest_asyncio
nest_asyncio.apply()

client = AsyncClient()

#retrieval results
src_paths = [
    "",
    ""
]
tgt_filenames = ["medexqa_qwen2.5.jsonl", "mmlu_qwen2.5.jsonl"]
model_name = "qwen2.5:7b"
output_dir = ""
concurrency = 2  # 并发数


os.makedirs(output_dir, exist_ok=True)

def extract_context(docs):
    if not docs:
        return ""
    context_parts = []
    for doc in docs:
        text = doc.get('text', '').strip()
        if text:
            context_parts.append(text)
    return "\n\n".join(context_parts)

def build_prompt(q, context="", max_context_length=4096):

    if context:
        context = context[:max_context_length] 
        return f"Based on the following medical documents:\n\n{context}\n\nAnswer the following question:\n\n{q}\n\nSelect only one option."
    else:
        return f"Answer the following medical question based on common knowledge:\n\n{q}\n\nSelect only one option."

async def chat_with_llama_async(prompt, temperature=0, num_predict=50, top_p=1.0):
    try:
        response = await client.chat(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": temperature,
                "num_predict": num_predict,  
                "top_p": top_p,
                "top_k": 1,
                "num_ctx": 8192, 
                "stream": False
            }
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"API call error: {e}")
        return ""

async def process_batch(batch_items):
    tasks = []
    for item in batch_items:
        docs = item.get("docs", [])
        context = extract_context(docs)  
        prompt = build_prompt(item["question"], context)  
        task = chat_with_llama_async(prompt)
        tasks.append((item, task))
    
    results = await asyncio.gather(*[task for _, task in tasks], return_exceptions=True)
    processed = []
    for (item, _), result in zip(tasks, results):
        model_answer = "" if isinstance(result, Exception) else result
        processed.append({
            "question": item["question"],
            "model_answer": model_answer,
            "ground_truths": item["ground_truths"]
        })
    return processed

async def process_dataset(src_path, tgt_path, total_lines):
    batch_items = []
    pbar = tqdm(total=total_lines, desc=f"Processing {os.path.basename(src_path)} with {model_name}", unit="lines")
    
    with open(src_path, "r", encoding="utf-8") as fin, \
         open(tgt_path, "w", encoding="utf-8") as fout:
        
        line_count = 0
        for line in fin:
            try:
                item = json.loads(line.strip())
                batch_items.append(item)
                line_count += 1
                
                if len(batch_items) == concurrency:
                    batch_results = await process_batch(batch_items)
                    for res in batch_results:
                        fout.write(json.dumps(res, ensure_ascii=False) + "\n")
                    batch_items.clear()
                    
                    elapsed = pbar.format_dict.get('elapsed', 0)
                    speed = line_count / elapsed if elapsed > 0 else 0
                    eta_min = (total_lines - line_count) / speed / 60 if speed > 0 else float('inf')
                    pbar.set_postfix({
                        "concurrency": concurrency,
                        "speed": f"{speed:.1f} lines/s",
                        "ETA": f"{eta_min:.1f} min" if eta_min != float('inf') else "N/A"
                    })
                    pbar.update(concurrency)
                
            except Exception as e:
                print(f"Skip line: {e}")
                line_count += 1
                pbar.update(1)
                continue
        
        if batch_items:
            batch_results = await process_batch(batch_items)
            for res in batch_results:
                fout.write(json.dumps(res, ensure_ascii=False) + "\n")
            pbar.update(len(batch_results))
    
    pbar.close()

async def main():
    for i, src_path in enumerate(src_paths):
        tgt_filename = tgt_filenames[i]
        tgt_path = os.path.join(output_dir, tgt_filename)
        
        with open(src_path, "r", encoding="utf-8") as fin_count:
            total_lines = sum(1 for _ in fin_count)
        
        await process_dataset(src_path, tgt_path, total_lines)
        print(f"Completed {src_path} -> {tgt_path} (Total: {total_lines} lines)")


await main()
